# computer_vision

`POST /computer_vision/`

One endpoint for all three computer vision services. `service` selects the pipeline:

| `service` | Does |
|---|---|
| `survivability_detection` | mangrove detection + alive/dead/unclear classification |
| `content_tagging` | tags photo evidence for L3 verification (`people`, `meterstick`, ...) |
| `content_moderation` | flags images needing human review |

`image_url` takes either a bare **S3 object key** or an **`https://` URL** to the same
object. The image is read and decoded once per request whichever service runs, and is
never written back.

**Needs:** the service running, `AWS_SOURCE_*` for the bucket, and `ANTHROPIC_API_KEY`
for the two tagging services.

In [ ]:
import json
import os
import sys

import requests
from dotenv import load_dotenv

# Run against a locally started service:  python3 src/main.py pipeline pipeline_api
sys.path.insert(0, os.path.abspath(".."))
load_dotenv(os.path.join("..", ".env"))

BASE_URL = os.getenv("VT_API_BASE_URL", "http://localhost:8000")
TOKEN = os.getenv("API_ENDPOINT_TOKEN")
HEADERS = {"Token": TOKEN}
ROUTE = f"{BASE_URL}/computer_vision/"

assert TOKEN, "API_ENDPOINT_TOKEN missing -- copy env_template to .env and fill it in"
print(f"target: {ROUTE}")

In [ ]:
# Confirm the service is up before sending anything to the route.
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5)
    print(health.status_code, health.json())
except requests.exceptions.ConnectionError:
    print("Service is not running. Start it with:\n"
          "    python3 src/main.py pipeline pipeline_api")

## The image

Set this once — every service below runs against it.

In [ ]:
IMAGE_URL = "bulk_uploads/<org_id>/<date>/<filename>.jpg"   # <-- S3 key, or an https:// URL


def run(service, image_url=None):
    """POST one service and return the parsed body (or print the error)."""
    payload = {"service": service, "image_url": image_url or IMAGE_URL}
    r = requests.post(ROUTE, json=payload, headers=HEADERS, timeout=180)
    if not r.ok:
        print(f"{service}: {r.status_code}\n{r.text[:500]}")
        return None
    return r.json()


## `survivability_detection`

Returns bounding boxes and per-detection probabilities so the client renders its own overlay.

In [ ]:
survivability = run("survivability_detection")

if survivability:
    print(json.dumps(survivability["result"]["counts"], indent=2))
    print(f"\nimage      : {survivability['image']}")
    print(f"detections : {len(survivability['result']['detections'])}")
    print(f"\nfirst box:\n{json.dumps(survivability['result']['detections'][0], indent=2)}")


### Draw the boxes

`bbox_xyxy` is in absolute pixels of the source image, in the frame given by `image`, so it can be drawn directly.

In [ ]:
from PIL import Image, ImageDraw

LOCAL_IMAGE = ""   # <-- path to a local copy of the same image

if survivability and LOCAL_IMAGE and os.path.exists(LOCAL_IMAGE):
    colours = {"alive": (0, 200, 0), "dead": (220, 0, 0), "unclear": (240, 180, 0)}

    image = Image.open(LOCAL_IMAGE).convert("RGB")
    draw = ImageDraw.Draw(image)
    for det in survivability["result"]["detections"]:
        x1, y1, x2, y2 = det["bbox_xyxy"]
        draw.rectangle([x1, y1, x2, y2], outline=colours[det["status"]], width=3)

    display(image.resize((image.width // 2, image.height // 2)))
else:
    print("Set LOCAL_IMAGE to a local copy of the source photo to render the overlay.")

## `content_tagging`

When `people` is detected the image is routed through moderation automatically and those tags are merged in — so a tagging call can return moderation tags too.

In [ ]:
tagging = run("content_tagging")

if tagging:
    print("tags:", tagging["result"]["tags"])
    for tag, score in sorted((tagging["result"].get("scores") or {}).items(), key=lambda kv: -kv[1]):
        print(f"  {tag:20s} {score}")

## `content_moderation`

A triage layer, never the final decision — a flagged image goes to a trained reviewer. Threshold and prompt live in `configs/config_cv_content_moderation.yaml`.

In [ ]:
moderation = run("content_moderation")

if moderation:
    result = moderation["result"]
    print("FLAGGED FOR REVIEW" if result["flagged"] else "not flagged")
    print("tags  :", result["tags"])
    print("scores:", json.dumps(result.get("scores") or {}, indent=2))

### Threshold sensitivity

`scores` comes back regardless of the configured threshold, so you can see what a different operating point would have flagged without redeploying.

In [ ]:
if moderation:
    scores = moderation["result"].get("scores") or {}
    for threshold in [0.1, 0.2, 0.3, 0.5, 0.8]:
        would_flag = [t for t, s in scores.items() if s > threshold]
        print(f"  threshold {threshold:<4} -> {would_flag or 'nothing flagged'}")

## The shared envelope

Every service returns the same outer fields; `service` tells you which `result` shape is inside.

In [ ]:
for body in [survivability, tagging, moderation]:
    if body:
        print(f"{body['service']:26s} envelope={sorted(body)}  result keys={sorted(body['result'])}")

## Error cases worth checking

- an unknown `service` -> **422**, rejected on the schema before any image is fetched
- a key that does not exist -> **502** (upstream storage failure)
- an object that is not an image -> **400** (fetched fine, wrong thing)
- omitting `service` or `image_url` -> **422**
- no `Token` header -> **401**

In [ ]:
checks = [
    ("unknown service ", {"service": "survivability", "image_url": IMAGE_URL}, HEADERS),
    ("missing key     ", {"service": "content_tagging", "image_url": "does/not/exist.jpg"}, HEADERS),
    ("no service      ", {"image_url": IMAGE_URL}, HEADERS),
    ("no image_url    ", {"service": "content_tagging"}, HEADERS),
    ("no auth token   ", {"service": "content_tagging", "image_url": IMAGE_URL}, {}),
]

for label, body, headers in checks:
    r = requests.post(ROUTE, json=body, headers=headers, timeout=60)
    print(f"{label} -> {r.status_code}")